# LeRobot ACT: Tutorial

이 Colab 튜토리얼은 Hugging Face의
[`training-act.ipynb`](https://github.com/huggingface/notebooks/blob/main/lerobot/training-act.ipynb)를
출발점으로 삼지만, **새로운 학습은 수행하지 않습니다.**

공개 ALOHA 데이터셋과 사전학습 ACT 정책으로 다음 실험을 진행합니다.

1. LeRobotDataset 메타데이터와 feature schema 읽기
2. 에피소드 이미지, 상태, 행동 궤적 시각화
3. 데이터 품질과 state-action 시간 정렬 진단
4. `delta_timestamps`로 ACT의 action chunk 직접 만들기
5. temporal ensemble을 합성 예측으로 재현하기
6. 이미지 증강과 가림(occlusion)에 대한 정책 민감도 확인
7. 사전학습 ACT의 offline action chunk 추론과 demonstration 비교
8. ACT action queue의 "한 번 추론, 여러 step 실행" 동작 측정

> 권장 런타임: **Colab GPU(T4 이상)**  
> 다운로드: 데이터 약 67 MB + 모델 약 203 MB  
> 주의: offline action MAE는 폐루프 성공률을 대신하지 않습니다.


## 0. Colab 준비

현재 LeRobot API를 고정된 커밋으로 설치합니다. 재현성을 위해 커밋을 명시했으며,
노트북을 최신 `main`에 맞추려면 `LEROBOT_COMMIT`만 변경하면 됩니다.


In [ ]:
# @title LeRobot 및 분석 패키지 설치
import pathlib
import subprocess
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"이 튜토리얼은 Python 3.12 이상이 필요합니다. 현재 버전: {sys.version}"
    )

LEROBOT_COMMIT = "8515d456be1dbef8c133f07188c785e683eca899"  # 2026-06-13
REPO_DIR = pathlib.Path("/content/lerobot")

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "https://github.com/huggingface/lerobot.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", LEROBOT_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", LEROBOT_COMMIT], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{REPO_DIR}[dataset]",
        "matplotlib>=3.8",
        "pandas>=2.0",
        "seaborn>=0.13",
        "imageio>=2.34",
    ],
    check=True,
)

print("LeRobot commit:", LEROBOT_COMMIT)


## 1. 공통 설정

아래 두 ID를 바꾸면 다른 ACT 데이터셋과 정책에도 같은 분석을 적용할 수 있습니다.
데이터셋과 정책의 카메라 key, state 차원, action 차원이 서로 일치해야 합니다.


In [ ]:
# @title Import 및 실험 설정
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision.transforms.functional as TVF
from IPython.display import HTML, display
from matplotlib import animation

from lerobot.datasets import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.policies import make_pre_post_processors
from lerobot.policies.act import ACTConfig, ACTPolicy

DATASET_ID = "lerobot/aloha_sim_transfer_cube_human"
MODEL_ID = "lerobot/act_aloha_sim_transfer_cube_human"
EPISODE_INDEX = 0
VIDEO_BACKEND = "pyav"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set_theme(style="whitegrid", context="notebook")
print("device:", DEVICE)
print("torch:", torch.__version__)


## 2. 데이터 전체를 받기 전에 메타데이터부터 읽기

`LeRobotDatasetMetadata`는 영상 전체를 내려받지 않고 FPS, episode 수, 카메라,
feature shape, 통계량을 확인할 수 있습니다. 새 데이터셋을 받았을 때 가장 먼저 실행할 셀입니다.


In [ ]:
meta = LeRobotDatasetMetadata(DATASET_ID)
camera_key = meta.camera_keys[0]

print(f"Dataset: {DATASET_ID}")
print(f"Episodes: {meta.total_episodes}")
print(f"Frames: {meta.total_frames}")
print(f"FPS: {meta.fps}")
print(f"Robot type: {meta.robot_type}")
print(f"Camera keys: {meta.camera_keys}")
print(f"Tasks: {meta.tasks}")

feature_rows = []
for key, spec in meta.features.items():
    feature_rows.append(
        {
            "key": key,
            "dtype": spec.get("dtype"),
            "shape": spec.get("shape"),
            "names": spec.get("names"),
        }
    )
display(pd.DataFrame(feature_rows))


### 체크 포인트

- ACT 입력: `observation.images.*`, `observation.state`
- ACT 출력: `action`
- 이 예제는 50 Hz, 14차원 양팔 관절, top camera 1개를 사용합니다.
- 자신의 데이터에서 feature 이름이나 shape가 다르면 모델을 그대로 사용할 수 없습니다.


## 3. 숫자 데이터와 영상 에피소드 불러오기

숫자 통계용 데이터셋은 `download_videos=False`로 빠르게 받고,
시각화용 데이터셋은 선택한 에피소드의 영상만 활성화합니다.


In [ ]:
# 전체 숫자 데이터: state/action/episode index 분석용
numeric_ds = LeRobotDataset(DATASET_ID, download_videos=False)

# 선택한 에피소드: 이미지 디코딩과 정책 입력용
episode_ds = LeRobotDataset(
    DATASET_ID,
    episodes=[EPISODE_INDEX],
    download_videos=True,
    video_backend=VIDEO_BACKEND,
)

print(numeric_ds)
print(episode_ds)


In [ ]:
# @title 분석 helper
def as_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def stack_column(column):
    return np.stack([as_numpy(value) for value in column])


def image_to_hwc(image):
    image = as_numpy(image)
    if image.shape[0] in (1, 3, 4):
        image = np.moveaxis(image, 0, -1)
    return np.clip(image, 0, 1)


def sync_if_needed():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


## 4. 한 frame의 모든 modality 확인

로봇 데이터는 이미지 하나가 아니라 같은 timestamp의 상태와 행동을 함께 확인해야 합니다.


In [ ]:
middle_index = len(episode_ds) // 2
sample = episode_ds[middle_index]

sample_summary = []
for key, value in sample.items():
    shape = tuple(value.shape) if hasattr(value, "shape") else None
    sample_summary.append({"key": key, "type": type(value).__name__, "shape": shape})
display(pd.DataFrame(sample_summary))

fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(image_to_hwc(sample[camera_key]))
ax.set_title(f"Episode {EPISODE_INDEX}, local frame {middle_index}, {camera_key}")
ax.axis("off")
plt.show()

print("state[:6] =", sample["observation.state"][:6])
print("action[:6] =", sample["action"][:6])


## 5. 에피소드 filmstrip과 짧은 애니메이션

성공 demonstration인지, 카메라가 흔들리는지, reset 구간이 섞였는지 먼저 눈으로 확인합니다.


In [ ]:
filmstrip_indices = np.linspace(0, len(episode_ds) - 1, 8, dtype=int)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, idx in zip(axes.flat, filmstrip_indices):
    frame = episode_ds[int(idx)]
    ax.imshow(image_to_hwc(frame[camera_key]))
    ax.set_title(f"t={idx / meta.fps:.2f}s")
    ax.axis("off")

plt.suptitle(f"Episode {EPISODE_INDEX} filmstrip", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# @title 선택한 에피소드 애니메이션
animation_indices = np.linspace(0, len(episode_ds) - 1, 40, dtype=int)
animation_frames = [image_to_hwc(episode_ds[int(i)][camera_key]) for i in animation_indices]

fig, ax = plt.subplots(figsize=(8, 6))
artist = ax.imshow(animation_frames[0])
title = ax.set_title("")
ax.axis("off")

def update_animation(frame_number):
    artist.set_data(animation_frames[frame_number])
    idx = animation_indices[frame_number]
    title.set_text(f"Episode {EPISODE_INDEX} | t={idx / meta.fps:.2f}s")
    return artist, title

anim = animation.FuncAnimation(
    fig,
    update_animation,
    frames=len(animation_frames),
    interval=120,
    blit=False,
)
plt.close(fig)
HTML(anim.to_jshtml())


## 6. 전체 state/action 배열 만들기

영상 디코딩 없이 전체 20,000 frame의 숫자 feature를 배열로 바꿉니다.


In [ ]:
raw_table = numeric_ds.hf_dataset

states = stack_column(raw_table["observation.state"]).astype(np.float32)
actions = stack_column(raw_table["action"]).astype(np.float32)
episode_indices = stack_column(raw_table["episode_index"]).reshape(-1).astype(int)
frame_indices = stack_column(raw_table["frame_index"]).reshape(-1).astype(int)
timestamps = stack_column(raw_table["timestamp"]).reshape(-1).astype(float)

motor_names = meta.features["action"].get("names", {}).get("motors")
if motor_names is None:
    motor_names = [f"action_{i}" for i in range(actions.shape[1])]

print("states:", states.shape)
print("actions:", actions.shape)
print("episode_indices:", episode_indices.shape)


## 7. 상태와 행동 궤적 비교

observation state와 command action 사이의 차이, gripper 전환 시점,
불연속적인 spike를 확인합니다.


In [ ]:
episode_mask = episode_indices == EPISODE_INDEX
ep_states = states[episode_mask]
ep_actions = actions[episode_mask]
ep_time = np.arange(len(ep_actions)) / meta.fps

selected_dims = [0, 1, 6, 7, 8, 13]
fig, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)

for ax, dim in zip(axes.flat, selected_dims):
    ax.plot(ep_time, ep_states[:, dim], label="observation.state", linewidth=1.6)
    ax.plot(ep_time, ep_actions[:, dim], label="action", linewidth=1.2, alpha=0.85)
    ax.set_title(f"{dim}: {motor_names[dim]}")
    ax.set_ylabel("position / command")

axes[-1, 0].set_xlabel("time [s]")
axes[-1, 1].set_xlabel("time [s]")
axes[0, 0].legend()
plt.tight_layout()
plt.show()


## 8. 에피소드별 데이터 품질 점검

아래 지표는 실패 여부를 자동 판정하지는 않지만, 다른 episode와 크게 다른 궤적을 찾는 데 유용합니다.

- `action_range_mean`: 각 action dimension의 범위를 평균
- `speed_rms`: frame 간 action 변화량의 RMS
- `total_variation`: 전체 action 변화량
- `outlier_score`: 위 변화량 지표들의 간단한 z-score 합


In [ ]:
quality_rows = []
for ep in np.unique(episode_indices):
    a = actions[episode_indices == ep]
    da = np.diff(a, axis=0)
    quality_rows.append(
        {
            "episode": int(ep),
            "frames": len(a),
            "duration_s": len(a) / meta.fps,
            "action_range_mean": np.ptp(a, axis=0).mean(),
            "speed_rms": np.sqrt(np.mean(da**2)),
            "total_variation": np.abs(da).sum(),
        }
    )

quality_df = pd.DataFrame(quality_rows)
for column in ["action_range_mean", "speed_rms", "total_variation"]:
    std = quality_df[column].std()
    quality_df[f"{column}_z"] = (
        (quality_df[column] - quality_df[column].mean()) / std if std > 0 else 0
    )

quality_df["outlier_score"] = quality_df[
    ["action_range_mean_z", "speed_rms_z", "total_variation_z"]
].abs().sum(axis=1)

display(quality_df.sort_values("outlier_score", ascending=False).head(10))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, column in zip(
    axes,
    ["action_range_mean", "speed_rms", "total_variation"],
):
    sns.scatterplot(data=quality_df, x="episode", y=column, ax=ax)
    ax.set_title(column)
plt.tight_layout()
plt.show()


## 9. state-action timestamp 정렬 진단

`state[t]`와 `action[t + lag]`의 상관을 비교합니다. peak가 0에서 멀면
기록 파이프라인의 지연이나 timestamp mismatch를 의심할 수 있습니다.
다만 로봇 dynamics 자체의 지연도 포함되므로 절대적인 판정 기준은 아닙니다.


In [ ]:
def lagged_correlation(state_signal, action_signal, lag):
    if lag > 0:
        x, y = state_signal[:-lag], action_signal[lag:]
    elif lag < 0:
        x, y = state_signal[-lag:], action_signal[:lag]
    else:
        x, y = state_signal, action_signal
    if np.std(x) < 1e-8 or np.std(y) < 1e-8:
        return np.nan
    return np.corrcoef(x, y)[0, 1]


lag_dim = 1
lags = np.arange(-20, 21)
correlations = np.array(
    [lagged_correlation(ep_states[:, lag_dim], ep_actions[:, lag_dim], lag) for lag in lags]
)
best_lag = int(lags[np.nanargmax(np.abs(correlations))])

plt.figure(figsize=(10, 4))
plt.plot(lags, correlations, marker="o")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.axvline(best_lag, color="crimson", linestyle=":", label=f"peak lag={best_lag}")
plt.xlabel("lag [frames]: corr(state[t], action[t+lag])")
plt.ylabel("correlation")
plt.title(f"State-action lag diagnostic: {motor_names[lag_dim]}")
plt.legend()
plt.show()

print(f"peak lag: {best_lag} frames = {best_lag / meta.fps:.3f} s")


## 10. `delta_timestamps`로 ACT action chunk 만들기

ACT는 한 시점에서 다음 `chunk_size`개의 action을 목표로 사용합니다.
LeRobotDataset은 미래 timestamp를 요청하면 자동으로 `[chunk, action_dim]`을 반환하고,
episode 끝을 넘어간 항목은 마지막 action으로 padding한 뒤 `action_is_pad`로 표시합니다.


In [ ]:
CHUNK_SIZE = 100
action_deltas = [step / meta.fps for step in range(CHUNK_SIZE)]

chunk_ds = LeRobotDataset(
    DATASET_ID,
    episodes=[EPISODE_INDEX],
    delta_timestamps={"action": action_deltas},
    download_videos=True,
    video_backend=VIDEO_BACKEND,
)

anchor_index = min(150, len(chunk_ds) - 1)
chunk_sample = chunk_ds[anchor_index]
target_chunk = as_numpy(chunk_sample["action"])
target_pad = as_numpy(chunk_sample["action_is_pad"]).astype(bool)

print("action chunk shape:", target_chunk.shape)
print("padding count at anchor:", int(target_pad.sum()))

near_end = chunk_ds[len(chunk_ds) - 10]
print("padding count near episode end:", int(near_end["action_is_pad"].sum()))


In [ ]:
chunk_time = np.arange(CHUNK_SIZE) / meta.fps
chunk_dims = [0, 1, 6, 7, 8, 13]

fig, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)
for ax, dim in zip(axes.flat, chunk_dims):
    ax.plot(chunk_time, target_chunk[:, dim], linewidth=1.7)
    ax.set_title(f"{dim}: {motor_names[dim]}")
    ax.set_ylabel("target action")
axes[-1, 0].set_xlabel("future time [s]")
axes[-1, 1].set_xlabel("future time [s]")
plt.suptitle(f"Action chunk from local frame {anchor_index}")
plt.tight_layout()
plt.show()

episode_steps = len(chunk_ds)
high_level_decisions = math.ceil(episode_steps / CHUNK_SIZE)
print(f"single-step policy decisions: {episode_steps}")
print(f"idealized chunk-level decisions: {high_level_decisions}")
print(f"effective horizon reduction: about {episode_steps / high_level_decisions:.1f}x")


## 11. Temporal ensemble을 합성 예측으로 재현

매 timestep마다 겹치는 chunk를 새로 예측한다고 가정하고,
같은 미래 시점에 대한 여러 예측을 지수 가중 평균합니다.
ACT 구현의 양수 계수는 오래된 예측에 조금 더 큰 가중치를 줍니다.


In [ ]:
def temporal_ensemble_offline(chunks, coefficient=0.01):
    # chunks[q, k, d] = query q가 global time q+k에 대해 낸 예측
    num_queries, chunk_size, action_dim = chunks.shape
    horizon = num_queries + chunk_size - 1
    ensembled = np.full((horizon, action_dim), np.nan, dtype=np.float32)

    for global_t in range(horizon):
        query_start = max(0, global_t - chunk_size + 1)
        query_end = min(num_queries - 1, global_t)
        predictions = []
        for query in range(query_start, query_end + 1):
            local_step = global_t - query
            predictions.append(chunks[query, local_step])

        predictions = np.stack(predictions)
        weights = np.exp(-coefficient * np.arange(len(predictions), dtype=np.float32))
        weights /= weights.sum()
        ensembled[global_t] = (predictions * weights[:, None]).sum(axis=0)

    return ensembled


rng = np.random.default_rng(SEED)
demo = ep_actions[:220]
synthetic_chunk_size = 40
num_queries = len(demo) - synthetic_chunk_size + 1
synthetic_chunks = np.stack(
    [
        demo[q : q + synthetic_chunk_size]
        + rng.normal(0, 0.035, size=(synthetic_chunk_size, demo.shape[1]))
        + 0.012 * np.sin(q / 8)
        for q in range(num_queries)
    ]
).astype(np.float32)

ensembled = temporal_ensemble_offline(synthetic_chunks, coefficient=0.01)

# 비교 기준: 각 global time에서 가장 최근 query의 예측
latest_prediction = np.stack(
    [synthetic_chunks[min(t, num_queries - 1), max(0, t - (num_queries - 1))] for t in range(len(demo))]
)

valid_length = min(len(demo), len(ensembled))
dim = 1
latest_mae = np.abs(latest_prediction[:valid_length] - demo[:valid_length]).mean()
ensemble_mae = np.abs(ensembled[:valid_length] - demo[:valid_length]).mean()

plt.figure(figsize=(14, 5))
plt.plot(demo[:valid_length, dim], label="demonstration", linewidth=2)
plt.plot(latest_prediction[:valid_length, dim], label="latest chunk prediction", alpha=0.55)
plt.plot(ensembled[:valid_length, dim], label="temporal ensemble", linewidth=1.7)
plt.title(f"Synthetic temporal ensemble: {motor_names[dim]}")
plt.xlabel("global frame")
plt.legend()
plt.show()

print(f"latest prediction MAE: {latest_mae:.5f}")
print(f"temporal ensemble MAE: {ensemble_mae:.5f}")


## 12. 이미지 증강을 직접 보고 통계 비교

증강은 데이터를 새로 기록하지 않고 조명과 가림 변화에 대한 robustness를 실험하는 방법입니다.
아래 셀은 학습하지 않고 입력 변화만 시각화합니다.


In [ ]:
base_image = sample[camera_key].clone()
occluded_image = base_image.clone()
_, height, width = occluded_image.shape
occluded_image[:, height // 3 : 2 * height // 3, width // 3 : 2 * width // 3] = 0

image_variants = {
    "original": base_image,
    "brightness x1.5": TVF.adjust_brightness(base_image, 1.5),
    "contrast x1.7": TVF.adjust_contrast(base_image, 1.7),
    "saturation x0.3": TVF.adjust_saturation(base_image, 0.3),
    "gaussian blur": TVF.gaussian_blur(base_image, kernel_size=[15, 15], sigma=[3.0, 3.0]),
    "center occlusion": occluded_image,
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
variant_stats = []
for ax, (name, image) in zip(axes.flat, image_variants.items()):
    ax.imshow(image_to_hwc(image))
    ax.set_title(name)
    ax.axis("off")
    variant_stats.append(
        {
            "variant": name,
            "mean": float(image.mean()),
            "std": float(image.std()),
            "min": float(image.min()),
            "max": float(image.max()),
        }
    )
plt.tight_layout()
plt.show()
display(pd.DataFrame(variant_stats))


## 13. 사전학습 ACT 불러오기

이 공개 체크포인트는 구형 LeRobot에서 학습되어 normalization buffer가 모델 파일 안에 들어 있습니다.
현재 LeRobot은 normalization을 processor로 분리하므로 다음 순서로 사용합니다.

1. ACT 가중치를 `strict=False`로 로드
2. 데이터셋의 `meta.stats`로 pre/post processor 생성
3. 입력 정규화 → ACT chunk 추론 → action 단위로 역정규화


In [ ]:
policy_config = ACTConfig.from_pretrained(MODEL_ID)
policy_config.device = DEVICE
policy_config.use_amp = False

policy = ACTPolicy.from_pretrained(
    MODEL_ID,
    config=policy_config,
    strict=False,
)
policy.eval()

preprocessor, postprocessor = make_pre_post_processors(
    policy.config,
    dataset_stats=meta.stats,
)

total_params = sum(parameter.numel() for parameter in policy.parameters())
trainable_params = sum(parameter.numel() for parameter in policy.parameters() if parameter.requires_grad)

print("policy:", type(policy).__name__)
print("chunk_size:", policy.config.chunk_size)
print("n_action_steps:", policy.config.n_action_steps)
print("input features:", policy.config.input_features)
print("output features:", policy.config.output_features)
print(f"parameters: {total_params / 1e6:.2f} M")
print(f"trainable flags currently set on: {trainable_params / 1e6:.2f} M")


## 14. 한 frame에서 action chunk 추론

demonstration의 미래 action chunk와 정책 예측을 비교합니다.
이 값은 학습 데이터의 한 frame에 대한 reconstruction 성격의 진단이며,
실제 rollout 성공률은 아닙니다.


In [ ]:
if policy.config.chunk_size != CHUNK_SIZE:
    raise ValueError(
        f"Notebook CHUNK_SIZE={CHUNK_SIZE}, model chunk_size={policy.config.chunk_size}"
    )

raw_observation = {
    key: chunk_sample[key]
    for key in policy.config.input_features
}
processed_observation = preprocessor(raw_observation)

sync_if_needed()
started = time.perf_counter()
with torch.inference_mode():
    predicted_chunk_normalized = policy.predict_action_chunk(processed_observation)
sync_if_needed()
inference_ms = (time.perf_counter() - started) * 1000

predicted_chunk = postprocessor(predicted_chunk_normalized)[0].numpy()
valid_mask = ~target_pad
offline_mae = np.abs(predicted_chunk[valid_mask] - target_chunk[valid_mask]).mean()
per_dimension_mae = np.abs(predicted_chunk[valid_mask] - target_chunk[valid_mask]).mean(axis=0)

print("predicted chunk:", predicted_chunk.shape)
print(f"inference latency: {inference_ms:.1f} ms")
print(f"offline action MAE: {offline_mae:.6f}")
display(
    pd.DataFrame(
        {
            "dimension": np.arange(len(motor_names)),
            "motor": motor_names,
            "MAE": per_dimension_mae,
        }
    ).sort_values("MAE", ascending=False)
)


In [ ]:
compare_dims = [0, 1, 6, 7, 8, 13]
fig, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)

for ax, dim in zip(axes.flat, compare_dims):
    ax.plot(chunk_time, target_chunk[:, dim], label="demonstration", linewidth=2)
    ax.plot(chunk_time, predicted_chunk[:, dim], label="ACT prediction", linewidth=1.5)
    ax.set_title(f"{dim}: {motor_names[dim]} | MAE={per_dimension_mae[dim]:.4f}")
    ax.set_ylabel("action")

axes[-1, 0].set_xlabel("future time [s]")
axes[-1, 1].set_xlabel("future time [s]")
axes[0, 0].legend()
plt.suptitle("Predicted action chunk vs demonstration")
plt.tight_layout()
plt.show()


## 15. ACT 추론이 deterministic한지 확인

ACT의 VAE encoder는 학습 시 demonstration style을 latent `z`로 압축하지만,
추론 시에는 target action이 없으므로 zero latent를 사용합니다.
같은 입력을 반복하면 같은 chunk가 나오는지 확인합니다.


In [ ]:
with torch.inference_mode():
    repeat_a = policy.predict_action_chunk(processed_observation)
    repeat_b = policy.predict_action_chunk(processed_observation)

max_repeat_difference = (repeat_a - repeat_b).abs().max().item()
print("maximum difference between repeated predictions:", max_repeat_difference)


## 16. ACT action queue 시간 측정

`select_action()`은 queue가 비었을 때만 전체 chunk를 추론합니다.
이후 `n_action_steps` 동안은 queue에서 action을 하나씩 꺼냅니다.
첫 호출과 후속 호출의 시간을 비교하면 action chunking의 runtime 동작을 볼 수 있습니다.


In [ ]:
policy.reset()
select_times_ms = []
selected_actions = []

for step in range(8):
    sync_if_needed()
    started = time.perf_counter()
    with torch.inference_mode():
        action_normalized = policy.select_action(processed_observation)
    sync_if_needed()
    select_times_ms.append((time.perf_counter() - started) * 1000)
    selected_actions.append(postprocessor(action_normalized)[0].numpy())

timing_df = pd.DataFrame(
    {
        "step": np.arange(len(select_times_ms)),
        "select_action_ms": select_times_ms,
        "source": ["model forward"] + ["action queue"] * (len(select_times_ms) - 1),
    }
)
display(timing_df)

plt.figure(figsize=(9, 4))
sns.barplot(data=timing_df, x="step", y="select_action_ms", hue="source", dodge=False)
plt.yscale("log")
plt.title("ACT select_action latency: first inference vs queued actions")
plt.ylabel("milliseconds, log scale")
plt.show()


## 17. 카메라 가림에 대한 정책 민감도

같은 state에서 이미지 중앙만 가린 뒤 action chunk가 얼마나 바뀌는지 측정합니다.
이는 정식 explainability 기법은 아니지만, 카메라 입력 의존성을 빠르게 확인하는 smoke test입니다.


In [ ]:
masked_observation = {
    key: value.clone() if torch.is_tensor(value) else value
    for key, value in raw_observation.items()
}
masked_image = masked_observation[camera_key]
_, height, width = masked_image.shape
masked_image[:, height // 3 : 2 * height // 3, width // 3 : 2 * width // 3] = 0

masked_processed = preprocessor(masked_observation)
with torch.inference_mode():
    masked_prediction_normalized = policy.predict_action_chunk(masked_processed)
masked_prediction = postprocessor(masked_prediction_normalized)[0].numpy()

occlusion_sensitivity = np.abs(masked_prediction - predicted_chunk).mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].imshow(image_to_hwc(masked_observation[camera_key]))
axes[0].set_title("center-occluded policy input")
axes[0].axis("off")

axes[1].bar(np.arange(len(motor_names)), occlusion_sensitivity)
axes[1].set_xticks(np.arange(len(motor_names)))
axes[1].set_xticklabels(motor_names, rotation=80, ha="right")
axes[1].set_ylabel("mean |action change| over chunk")
axes[1].set_title("Action sensitivity to center occlusion")
plt.tight_layout()
plt.show()


## 18. 여러 시점의 offline 진단

에피소드의 여러 시점에서 같은 비교를 반복합니다. 샘플 수를 늘리면 시간이 더 걸립니다.
이 결과도 closed-loop evaluation이 아니라 demonstration action과의 offline 차이입니다.


In [ ]:
NUM_OFFLINE_CHECKS = 6
check_indices = np.linspace(20, len(chunk_ds) - 120, NUM_OFFLINE_CHECKS, dtype=int)
offline_rows = []

for idx in check_indices:
    item = chunk_ds[int(idx)]
    item_observation = {key: item[key] for key in policy.config.input_features}
    item_processed = preprocessor(item_observation)

    sync_if_needed()
    started = time.perf_counter()
    with torch.inference_mode():
        item_prediction_normalized = policy.predict_action_chunk(item_processed)
    sync_if_needed()

    item_prediction = postprocessor(item_prediction_normalized)[0].numpy()
    item_target = as_numpy(item["action"])
    item_valid = ~as_numpy(item["action_is_pad"]).astype(bool)
    item_mae = np.abs(item_prediction[item_valid] - item_target[item_valid]).mean()

    offline_rows.append(
        {
            "local_frame": int(idx),
            "time_s": idx / meta.fps,
            "MAE": float(item_mae),
            "latency_ms": (time.perf_counter() - started) * 1000,
        }
    )

offline_df = pd.DataFrame(offline_rows)
display(offline_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=offline_df, x="time_s", y="MAE", marker="o", ax=axes[0])
axes[0].set_title("Offline chunk MAE by episode time")
sns.barplot(data=offline_df, x="local_frame", y="latency_ms", ax=axes[1])
axes[1].set_title("Inference latency")
plt.tight_layout()
plt.show()


## 19. 결과 저장

분석 표를 CSV/JSON으로 저장합니다. Colab 왼쪽 파일 패널에서 내려받을 수 있습니다.


In [ ]:
output_dir = Path("/content/act_tutorial_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

quality_df.to_csv(output_dir / "episode_quality.csv", index=False)
offline_df.to_csv(output_dir / "offline_act_diagnostics.csv", index=False)

summary = {
    "dataset_id": DATASET_ID,
    "model_id": MODEL_ID,
    "episode_index": EPISODE_INDEX,
    "fps": meta.fps,
    "chunk_size": policy.config.chunk_size,
    "single_frame_offline_mae": float(offline_mae),
    "single_frame_inference_ms": float(inference_ms),
    "deterministic_repeat_max_diff": float(max_repeat_difference),
    "peak_state_action_lag_frames": int(best_lag),
}
(output_dir / "summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("saved to:", output_dir)
print(*sorted(path.name for path in output_dir.iterdir()), sep="\n")


## 20. 추가 실습 과제

1. `EPISODE_INDEX`를 바꾸고 outlier score가 큰 episode를 영상으로 확인한다.
2. temporal ensemble coefficient를 `0`, `0.01`, `0.1`, `-0.01`로 바꿔 결과를 비교한다.
3. 중앙 가림 대신 3×3 grid의 각 영역을 가려 action sensitivity heatmap을 만든다.
4. `lerobot/aloha_sim_insertion_human`과
   `lerobot/act_aloha_sim_insertion_human`으로 동일 분석을 반복한다.
5. 자신의 데이터셋과 ACT 체크포인트로 ID를 바꾸고 feature schema가 일치하는지 검사한다.
6. offline MAE가 낮지만 실제 성공률이 낮은 사례를 수집해 covariate shift를 토론한다.

## 공식 자료

- [원본 ACT 학습 노트북](https://github.com/huggingface/notebooks/blob/main/lerobot/training-act.ipynb)
- [LeRobotDataset v3 문서](https://huggingface.co/docs/lerobot/lerobot-dataset-v3)
- [LeRobot ACT 문서](https://huggingface.co/docs/lerobot/act)
- [공개 ALOHA transfer cube 데이터셋](https://huggingface.co/datasets/lerobot/aloha_sim_transfer_cube_human)
- [사전학습 ACT 체크포인트](https://huggingface.co/lerobot/act_aloha_sim_transfer_cube_human)
- [ACT 논문](https://arxiv.org/abs/2304.13705)
